In [2]:
import numpy as np
import random as random
import time as time
from itertools import combinations, chain
import joblib
from joblib import Parallel, delayed
import pickle
import math

In [3]:
#functions for defining partitions, categories

def categories(nums):
    result = []
    for r in range(1, len(nums) + 1):
        for subset in combinations(nums, r):
            result.append(list(subset))
    return result

def partitions(lst):
    if not lst:
        return [[]]

    result = []

    # Generate all partitions recursively
    def helper(partition, remaining):
        if not remaining:
            result.append(partition)
            return
        first = remaining[0]
        rest = remaining[1:]
        for i in range(len(partition)):
            new_partition = partition[:i] + [partition[i] + [first]] + partition[i+1:]
            helper(new_partition, rest)
        helper(partition + [[first]], rest)

    helper([], lst)
    return result

def cat_index(fstate, fpartition, fcat):
    for fpart in fpartition:
        if fstate in fpart:
            for i, subset in enumerate(fcat):
                if sorted(subset) == sorted(fpart):
                    return i
    return -1  # If x isn't found (or no matching subset found)

def save(data, filename):
    with open(filename, 'wb') as file:
        pickle.dump(data, file)

def load(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

In [10]:
#simulation 

def single_run(states, signals, acts, timesteps, rein, pun, cats, parts, state_part):

    #sender urns
    #part_urn = [1 for i in range(len(parts))]
    part_urn = np.ones(len(parts))
    cat_urns = np.ones((len(cats), signals))
        
    #receiver urns
    act_urns = np.ones((signals, acts))
    
    #success counter
    succ = np.zeros(timesteps)
    
    for t in range(timesteps):
        #state of nature determined
        state = np.random.choice([i for i in range(states)], p=[1/states for i in range(states)])
    
        #sender partition chosen, relevant category urn identified, signal chosen
        part_probs = part_urn / part_urn.sum()
        part_choice = np.random.choice(len(parts), p=part_probs)
        active_part = list.copy(parts[part_choice])
        active_cat = cat_index(state, active_part, cats)
        active_cat_urn = cat_urns[active_cat]
        signal_probs = [active_cat_urn[x]/sum(active_cat_urn) for x in range(signals)]
        signal = np.random.choice([i for i in range(signals)], p=signal_probs)
    
        #receiver act chosen
        act_probs = [act_urns[signal][x]/sum(act_urns[signal]) for x in range(acts)]
        act = np.random.choice([i for i in range(acts)], p=act_probs)
        
        if state in state_part[act]:
            #reinforce partition, signal conditional on category and state, act conditional on signal
            part_urn[part_choice] += rein
            cat_urns[active_cat][signal] += rein
            act_urns[signal][act] += rein
            succ[t] = 1
    
        else:
            #punish the same
            part_urn[part_choice] -= pun
            cat_urns[active_cat][signal] -= pun
            act_urns[signal][act] -= pun
    
            #prevent extinction
            if part_urn[part_choice] < 1:
                part_urn[part_choice] = 1
            if cat_urns[active_cat][signal] < 1:
                cat_urns[active_cat][signal] = 1
            if act_urns[signal][act] < 1:
                act_urns[signal][act] = 1
                
    return (
        part_urn / part_urn.sum(),
        cat_urns / cat_urns.sum(axis=1, keepdims=True),
        act_urns / act_urns.sum(axis=1, keepdims=True),
        succ
    )

In [11]:
#parameter settings
runs=100
rstates, rsignals, racts = 3, 2, 2
rtimesteps = 100000
rcats = categories([i for i in range(rstates)])
rparts = partitions([i for i in range(rstates)])
rstate_part = [[0],[1,2]]
#state_part defines payoff structure; state_part[i] contains all states where i is the optimal act

In [12]:

t1 = time.time()

results = (Parallel(n_jobs = 4)(delayed(single_run)(rstates, rsignals, racts, rtimesteps, 1, 1, rcats, rparts, rstate_part) for p in range(runs)))

t2 = time.time()

print(t2-t1)

96.41432046890259
